# Proyecto: Clasificador Inteligente de Imágenes de Ropa
## Evaluación del módulo 8: FUNDAMENTOS DE DEEP LEARNING

**Unidad solicitante:** Área de Ciencia de Datos de la tienda virtual StyleNet.  
**Objetivo del proyecto:** Diseñar y entrenar un modelo de Deep Learning capaz de clasificar imágenes de prendas de vestir de forma automática para optimizar la eficiencia operativa de la plataforma.

---

## Lección 1: LA RED NEURONAL ARTIFICIAL

**Objetivo:** Comprender los elementos de una red neuronal artificial y su rol en la resolución de problemas.

### A. Definir la arquitectura de una red neuronal densa

Para abordar el problema de clasificación de StyleNet, utilizaremos una **Red Neuronal Densa** (también conocida como Perceptrón Multicapa). Esta arquitectura es ideal para procesar datos donde cada característica de entrada puede estar relacionada con las demás para determinar la categoría final.

Los componentes fundamentales de nuestra arquitectura son:

* **Capa de Entrada (Flatten):** Como las imágenes son bidimensionales (matrices de píxeles), primero aplicamos una capa que "aplana" estos datos, convirtiéndolos en un vector unidimensional que la red pueda procesar.
* **Capas Ocultas (Dense):** Son capas con múltiples neuronas interconectadas. Cada neurona aplica pesos a las señales de entrada y utiliza una función de activación (generalmente **ReLU**) para introducir no linealidad, permitiendo que la red aprenda patrones complejos de las prendas.
* **Capa de Salida:** Contiene una neurona por cada categoría de ropa disponible. Se utiliza la función de activación **Softmax**, la cual entrega una distribución de probabilidad para que el modelo prediga la categoría con mayor nivel de confianza.

In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import datasets
import numpy as np

# Cargamos el dataset Fashion-MNIST
(train_images, train_labels), (test_images, test_labels) = datasets.fashion_mnist.load_data()

# Normalizamos los valores de los píxeles (de 0-255 a 0-1)
train_images = train_images / 255.0
test_images = test_images / 255.0

print("Datos cargados y normalizados con éxito.")



# Definimos la arquitectura del modelo para StyleNet
def crear_modelo_denso(input_shape, num_clases):
    model = models.Sequential([
        # 1. Capa de entrada: Aplanamos la imagen (ej: de 28x28 a 784 píxeles)
        layers.Input(shape=input_shape),
        layers.Flatten(),

        # 2. Capas ocultas: Neuronas que aprenden los patrones de la ropa
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),

        # 3. Capa de salida: Una neurona por cada categoría de prenda
        layers.Dense(num_clases, activation='softmax')
    ])
    return model

# Ejemplo de uso: imágenes de 28x28 píxeles y 10 categorías de ropa
modelo_stylenet = crear_modelo_denso(input_shape=(28, 28), num_clases=10)
modelo_stylenet.summary()

Datos cargados y normalizados con éxito.
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_2 (Flatten)         (None, 784)               0         
                                                                 
 dense_12 (Dense)            (None, 128)               100480    
                                                                 
 dense_13 (Dense)            (None, 64)                8256      
                                                                 
 dense_14 (Dense)            (None, 10)                650       
                                                                 
Total params: 109,386
Trainable params: 109,386
Non-trainable params: 0
_________________________________________________________________


## B. Implementar una red neuronal simple para clasificación binaria

En esta sección, construiremos una Red Neuronal Artificial (ANN) básica para resolver un problema de clasificación binaria. El objetivo es que el modelo aprenda a asignar una etiqueta (0 o 1) a partir de un conjunto de características de entrada.

### Componentes del modelo:
* **Capas Ocultas**: Utilizaremos funciones de activación **ReLU**, que permiten a la red aprender relaciones no lineales y patrones complejos en los datos.
* **Capa de Salida**: Una única neurona con activación **Sigmoide**, que transforma la salida en una probabilidad acotada entre 0 y 1.
* **Función de Pérdida**: `binary_crossentropy`, que es el estándar para medir el error en problemas donde solo existen dos clases posibles.

---

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import datasets
import numpy as np

# Cargamos el dataset Fashion-MNIST
(train_images, train_labels), (test_images, test_labels) = datasets.fashion_mnist.load_data()

# Normalizamos los valores de los píxeles (de 0-255 a 0-1)
train_images = train_images / 255.0
test_images = test_images / 255.0

print("Datos cargados y normalizados con éxito.")

# 1. Creación de datos sintéticos (Ejemplo: Edad y Puntaje de Crédito)
np.random.seed(42)
X = np.random.rand(1000, 2)
y = (X[:, 0] + X[:, 1] > 1).astype(int)  # Clase 1 si la suma es > 1

# 2. Definición de la arquitectura
model = models.Sequential([
    layers.Dense(16, activation='relu', input_shape=(2,)),  # Capa de entrada + oculta
    layers.Dense(8, activation='relu'),                     # Capa oculta adicional
    layers.Dense(1, activation='sigmoid')                   # Capa de salida binaria
])

# 3. Compilación
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 4. Entrenamiento
print("Iniciando el entrenamiento del modelo...")
model.fit(X, y, epochs=20, batch_size=32, verbose=0)

# 5. Evaluación
loss, accuracy = model.evaluate(X, y, verbose=0)
print(f"Precisión final del modelo: {accuracy:.4f}")

Datos cargados y normalizados con éxito.
Iniciando el entrenamiento del modelo...
Precisión final del modelo: 0.9660


### Análisis de los resultados

* **Eficacia del Modelo**: El valor de *accuracy* indica qué tan bien el modelo logró separar las dos clases. Al usar una regla clara en los datos sintéticos, la red converge rápidamente.
* **Función de Activación**: La sigmoide en la salida es la que permite interpretar el resultado como una probabilidad (ej. 0.8 = 80% de certeza de que es Clase 1).
* **Optimización**: El optimizador `Adam` ajusta los pesos de forma automática para minimizar la pérdida de manera eficiente.

## C. Identificar capas, pesos, funciones de activación y pérdida

Para entender cómo "aprende" realmente nuestra red neuronal, debemos ser capaces de inspeccionar sus componentes internos. Cada parte cumple una función crítica en el proceso de optimización:

1. **Capas (Layers):** Son los bloques de construcción. Tenemos la capa de entrada (datos), capas ocultas (extracción de características) y la capa de salida (predicción).
2. **Pesos ($w$) y Sesgos ($b$):** Son los parámetros ajustables. Los pesos determinan la fuerza de la conexión entre neuronas, y el sesgo permite desplazar la función de activación para ajustarse mejor a los datos.
3. **Funciones de Activación:** Introducen no linealidad. Sin ellas, una red neuronal sería solo una serie de operaciones lineales (una regresión simple).
4. **Función de Pérdida (Loss):** Es nuestra "brújula". Nos dice qué tan lejos está la predicción del valor real para que el optimizador sepa cuánto debe ajustar los pesos.

In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import datasets
import numpy as np

# Cargamos el dataset Fashion-MNIST
(train_images, train_labels), (test_images, test_labels) = datasets.fashion_mnist.load_data()

# Normalizamos los valores de los píxeles (de 0-255 a 0-1)
train_images = train_images / 255.0
test_images = test_images / 255.0

print("Datos cargados y normalizados con éxito.")

# Inspección de los componentes internos del modelo anterior
# NOTA: Esta celda usa la variable 'model' definida en la celda B.
# Asegúrate de ejecutar las celdas en orden.

print("--- Resumen de la Arquitectura ---")
model.summary()

print("\n--- Inspección de Parámetros (Pesos y Sesgos) ---")
# Extraemos los pesos de la primera capa oculta
pesos, sesgos = model.layers[0].get_weights()

print(f"Configuración de la Capa: {model.layers[0].name}")
print(f"Función de activación: {model.layers[0].activation.__name__}")
print(f"Forma de los pesos (Weights shape): {pesos.shape}")
print(f"Ejemplo de pesos aprendidos:\n{pesos[:2]}")

print("\n--- Configuración de Entrenamiento ---")
print(f"Función de Pérdida: {model.loss}")
print(f"Optimizador: {type(model.optimizer).__name__}")

Datos cargados y normalizados con éxito.
--- Resumen de la Arquitectura ---
Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_15 (Dense)            (None, 16)                48        
                                                                 
 dense_16 (Dense)            (None, 8)                 136       
                                                                 
 dense_17 (Dense)            (None, 1)                 9         
                                                                 
Total params: 193
Trainable params: 193
Non-trainable params: 0
_________________________________________________________________

--- Inspección de Parámetros (Pesos y Sesgos) ---
Configuración de la Capa: dense_15
Función de activación: relu
Forma de los pesos (Weights shape): (2, 16)
Ejemplo de pesos aprendidos:
[[ 0.7628804   0.15898177 -0.38223928 -0.5755513   0.5269

### Análisis de los componentes identificados

* **Jerarquía de Capas:** El `model.summary()` nos permite ver la cantidad de parámetros entrenables. Es vital notar que a medida que agregamos capas o neuronas, la complejidad aumenta exponencialmente, lo que puede llevar al sobreajuste si no tenemos suficientes datos.
* **El rol de los Pesos:** Al observar la forma (shape) de los pesos, confirmamos que coinciden con las conexiones entre la entrada (2 dimensiones) y las neuronas de la capa (16 en este caso). Estos valores son los que el algoritmo de *Backpropagation* modifica en cada época.
* **Configuración del aprendizaje:** Al identificar el optimizador (Adam) y la pérdida (Sparse Categorical Crossentropy), confirmamos que el flujo está alineado con un problema de clasificación múltiple (10 clases de ropa).